## Result filter module - _Attention-Retrieval (AR)_ - Fine-tuning

### Initialization

In [1]:
import sys
import os

# if not in 'notebooks' directory, change to it
if not os.getcwd().endswith('Result-filter'):
	os.chdir('irat')
	os.chdir('notebooks')
	os.chdir('Result-filter')

os.environ['WANDB_DISABLED'] = 'true'  # disable Weights & Biases logging

### Load the model and dataset

In [ ]:
from AR_evaluate import evaluate_model  # from local file
from sentence_transformers import CrossEncoder

model_name = 'cross-encoder/ms-marco-MiniLM-L6-v2'  # 22.7M params
model_name_short = 'MiniLM-L6-v2'

# model_name = 'cross-encoder/ms-marco-MiniLM-L4-v2'  # 19.2M params
# model_name_short = 'MiniLM-L4-v2'

model = CrossEncoder(model_name)
# Base accuracy using R-Precision without applying k+1 or a threshold
base_accuracy = evaluate_model(model, model_name_short)

Accuracy: 78.71%
Model: MiniLM-L6-v2


### Prepare the data

In [3]:
# Preprocess the training dataset
# Create triplets of (query, passage, score) for training
import csv

train_data_file = 'coding_train_data.csv'

if os.path.exists(train_data_file):
	with open(train_data_file, 'r') as f:
		train_data = [row for row in csv.reader(f)][1:]
else:
	from datasets import load_from_disk
	dataset = load_from_disk('coding_dataset')
	train_dataset = dataset['train']
	validation_dataset = dataset['validation']

	train_data = []
	for val_index, row in enumerate(train_dataset):
		query = row['query']
		passages = row['passages']['passage_text']
		scores = row['passages']['is_selected']
		for passage, score in zip(passages, scores):
			train_data.append([query, passage, score])
	with open(train_data_file, 'w', newline='') as f:
		writer = csv.writer(f)
		writer.writerow(['query', 'passage', 'score'])  # Write header
		writer.writerows(train_data)

print('Sample:')
for query, passage, score in train_data[:1]:
	print(f'  Query: {query}  \n  Passage: {passage[:50]}...  \n  Score: {score}\n')
print(f'Train data size: {len(train_data)}')

Sample:
  Query: what is an of clause sql  
  Passage: SQL clauses site was designed to help programmers ...  
  Score: 0

Train data size: 25078


### Cross-encoder fine-tuning

In [4]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader
import torch

train_samples = [InputExample(texts=[query, passage], label=float(score))
				 for query, passage, score in train_data]

model_save_path = f'msmarco-coding-{model_name_short}'

try:
	if not os.path.exists(model_save_path):
		raise FileNotFoundError(f'Model not found: {model_save_path}')
	print('Trying to load the model...')
	loaded_model = CrossEncoder(model_save_path)
	if loaded_model:
		model = loaded_model
	print(f'Model loaded. Skipping fine-tuning.')
except KeyboardInterrupt:
	print('KeyboardInterrupt: Stopping fine-tuning.')
	sys.exit(0)
except:
	print('Loading the model...')
	model = CrossEncoder(model_name)
	print('Fine-tuning the model...')
	train_dataloader = DataLoader(train_samples,
							batch_size=32, shuffle=True)
	num_epochs = 3  # 5 for L4 model

	model.fit(
		train_dataloader=train_dataloader,
		epochs=num_epochs,
		loss_fct=torch.nn.BCEWithLogitsLoss(),  # works for binary labels
		warmup_steps=int(len(train_dataloader) * num_epochs * 0.1),  # 10% of total steps
		optimizer_class=torch.optim.AdamW,
		optimizer_params={ 'lr': 2e-5 },  # learning rate
		use_amp=True,  # for mixed precision training such as fp16
		output_path=None,  # skip saving every run
	)

ft_accuracy = evaluate_model(model, model_name_short+'-fine-tuned')
if ft_accuracy > base_accuracy:
	print(f'GOOD. Fine-tuned model is better than the base model.')
	model.save(model_save_path)
	print(f'Model saved to {model_save_path}')
else:
	print(f'BAD. Fine-tuned model is worse than the base model.')
print(f'                  ({base_accuracy*100:.2f} -> {ft_accuracy*100:.2f})')

Trying to load the model...
Model loaded. Skipping fine-tuning.
Accuracy: 78.87%
Model: MiniLM-L6-v2-fine-tuned
GOOD. Fine-tuned model is better than the base model.
Model saved to msmarco-coding-MiniLM-L6-v2
                  (78.71 -> 78.87)


### Hyperparameter tuning

In [5]:
# import csv
# import itertools
# import math
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch_optimizer

# result_filename = 'hyperparam_results.csv'
# try:
# 	results = pd.read_csv(result_filename).to_dict(orient='records')
# 	# replace 'None' with None in all columns
# 	for result in results:
# 		for key, value in result.items():
# 			if value == None or (type(value) == float and math.isnan(value)):
# 				result[key] = None
# except:
# 	results = []

# def save_last_result():
# 	with open(result_filename, mode='a', newline='') as file:
# 		writer = csv.DictWriter(file, fieldnames=results[0].keys())
# 		if len(results) == 1:  # we are saving first result. write header
# 			writer.writeheader()
# 		writer.writerow(results[-1])

# param_grid = {
# 	'learn_rates':    [1e-5, 2e-5, 3e-5],
# 	'batch_size':     [32],
# 	'epochs':         [2, 3, 5],
# 	'optimizers':     [torch.optim.AdamW, torch_optimizer.AdamP],
# 	'loss_functions': [None, nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.0]))]
# 						# Hyperparam tuning is to find best non-default parameters.
# }
# evaluated_combinations = {tuple(result.values())[:-1] for result in results}
# 												# -1 to exclude 'accuracy' values

# total_combinations = 1
# for key in param_grid:
# 	total_combinations *= len(param_grid[key])

# pending = 0
# for learn_rate, batch_size, epochs, optimizer, loss_fct in itertools.product(
# 		param_grid['learn_rates'], param_grid['batch_size'], param_grid['epochs'],
# 		param_grid['optimizers'], param_grid['loss_functions']):
# 	# check whether the current combination has already been evaluated
# 	loss_fct_str = str(loss_fct) if loss_fct else None
# 	current_combination = (learn_rate, batch_size, epochs, optimizer.__name__, loss_fct_str)
# 	if current_combination in evaluated_combinations:
# 		continue
# 	pending += 1

# print(f'Pending combinations: {pending}')


# for learn_rate, batch_size, epochs, optimizer, loss_fct in itertools.product(
# 		param_grid['learn_rates'], param_grid['batch_size'], param_grid['epochs'],
# 		param_grid['optimizers'], param_grid['loss_functions']):
# 	# check whether the current combination has already been evaluated
# 	loss_fct_str = str(loss_fct) if loss_fct else None
# 	current_combination = (learn_rate, batch_size, epochs, optimizer.__name__, loss_fct_str)
# 	if current_combination in evaluated_combinations:
# 		print('Skipping already evaluated combination.')
# 		continue

# 	print(f'Index:', len(results)+1, 'of', total_combinations)
# 	print(current_combination)
# 	pending -= 1
# 	print(f'Pending after this: {pending}')
# 	train_dl = DataLoader(train_samples, batch_size=batch_size, shuffle=True)

# 	model = CrossEncoder(model_name)
# 	model.fit(
# 		train_dataloader=train_dl,
# 		epochs=epochs,
# 		loss_fct=loss_fct,
# 		optimizer_class=optimizer,
# 		optimizer_params={
# 			'lr': learn_rate,
# 		},
# 		warmup_steps=int(0.1 * len(train_dl) * epochs),
# 		use_amp=True,
# 		output_path=None,  # skip saving every run
# 		show_progress_bar=False,
# 	)

# 	accuracy = evaluate_model(model, model_name_short=None)
# 	results.append({
# 		'learn_rate': learn_rate,
# 		'batch_size': batch_size,
# 		'epochs': epochs,
# 		'optimizer': optimizer.__name__,
# 		'loss_function': loss_fct_str,
# 		'accuracy': accuracy
# 	})
# 	save_last_result()
# 	print('-'*80)

# df = pd.DataFrame(results)
# df.sort_values(by='accuracy', ascending=False, inplace=True)
# df.fillna(value='None', inplace=True)
# df.to_csv(result_filename, index=False)